# 02 · GSE65391 · RNA_array · probes to genes

Reads the series matrix, `GPL10558.soft.gz` and NCBI gene_info. Writes `data/run_artifacts/GSE65391/genes.rds`.

1. Probes without an Entrez gene identifier (control probes) are dropped.
2. Several probes can measure one gene. One probe per gene is kept: the one with the highest mean intensity.
3. Genes are named with their current NCBI symbol, so all three studies use the same names.
4. A gene is marked **expressed** when its log2 intensity is above 6 in at least 10% of samples.
   The floor of this array's scale is log2(10) = 3.32.

In [1]:
source("../src/paths.R")
suppressMessages({library(GEOquery); library(Biobase)})
X   <- exprs(suppressMessages(getGEO(filename = raw("GSE65391", "GSE65391_series_matrix.txt.gz"), getGPL = FALSE)))
gpl <- Table(suppressMessages(getGEO(filename = raw("GSE65391", "GPL10558.soft.gz"))))
gi  <- read.delim(file.path(NCBI, "Homo_sapiens.gene_info.gz"), quote = "", colClasses = "character")
c(probes = nrow(X), samples = ncol(X), min = round(min(X), 2), max = round(max(X), 2))

probes  samples      min      max 
43799.00   996.00     1.92    15.65

In [2]:
entrez <- gpl$Entrez_Gene_ID[match(rownames(X), gpl$ID)]
entrez[!is.na(entrez) & !nzchar(entrez)] <- NA
ok <- !is.na(entrez) & rowSums(is.na(X)) == 0
Xok <- X[ok, ]; key <- entrez[ok]
brightest <- tapply(seq_len(nrow(Xok)), key, function(i) i[which.max(rowMeans(Xok[i, , drop = FALSE]))])
E <- Xok[unlist(brightest), ]
rownames(E) <- names(brightest)
symbol <- gi$Symbol[match(rownames(E), gi$GeneID)]
old    <- gpl$Symbol[match(rownames(E), gpl$Entrez_Gene_ID)]
symbol[is.na(symbol)] <- old[is.na(symbol)]         # identifiers retired since 2010 keep the array symbol
keep <- !duplicated(symbol)
E <- E[keep, ]; rownames(E) <- symbol[keep]
c(probes_with_entrez = sum(ok), genes = nrow(E))

probes_with_entrez              genes 
             40530              28948

In [3]:
expressed <- rowMeans(E > 6) >= 0.10
c(expressed_genes = sum(expressed))

expressed_genes 
           8825

**Result.** 40,530 probes carry an Entrez identifier and collapse to 28,948 genes; 8,825 are
expressed.

In [4]:
saveRDS(list(E = E, expressed = expressed), art("GSE65391", "genes.rds"))